![](https://github.com/arthurpaulino/miraiml/raw/master/docs/img/MiraiML.svg?sanitize=true) 

MiraiML: asynchronous, autonomous and continuous Machine Learning in Python https://miraiml.readthedocs.io


MiraiML
=======
    Mirai: `future` in japanese.

MiraiML is an asynchronous engine for continuous & autonomous machine learning,
built for real-time usage.
Usage
-----

1. Install: ``$ pip install miraiml``
2. Now, inside a Python environment, you can import the main components:

>>> from miraiml import SearchSpace, Config, Engine

You might want to `Read the Docs`_ for a better understanding of MiraiML.

Contributing
------------

Please, follow the guidelines_ if you want to be part of this project.

-  _examples: https://github.com/arthurpaulino/miraiml/tree/master/examples

- _Read the Docs: https://miraiml.readthedocs.io/en/latest/

- _guidelines: https://github.com/arthurpaulino/miraiml/blob/master/CONTRIBUTING.md

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.

/kaggle/input/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/house-prices-advanced-regression-techniques/test.csv
/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/house-prices-advanced-regression-techniques/data_description.txt


In [2]:
!pip install miraiml

  Created wheel for miraiml: filename=MiraiML-3.0.0-py3-none-any.whl size=18956 sha256=1abe0cebe0b02851b754553e951db33e5127e7800187032542f38bd754a7d583
  Stored in directory: /root/.cache/pip/wheels/7d/a9/13/20e79ba660f8fb9ca6b80b18ba009359b4221907afccdf1cff
Successfully built miraiml


In [3]:
# Read the data
data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
data = data[['LotArea', 'OverallQual', 'YearBuilt', 'TotRmsAbvGrd', 'SalePrice']]

In [4]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(data, test_size=0.2)

# Building the search spaces
Let's compare (and ensemble) a ``KNeighborsRegressor`` and a pipeline composed by ``QuantileTransformer`` and a ``LinearRegression``.

In [5]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import  LinearRegression
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import StandardScaler

from miraiml import SearchSpace
from miraiml.pipeline import compose

Pipeline = compose(
    [('scaler', StandardScaler), ('linear_reg', LinearRegression)]
)

search_spaces = [
    SearchSpace(
        id='k-NeighborsRegressor',
        model_class=KNeighborsRegressor,
        parameters_values=dict(
            n_neighbors=range(2, 9),
            weights=['uniform', 'distance'],
            p=range(2, 5)
        )
    ),
    SearchSpace(
        id='Pipeline',
        model_class=Pipeline,
        parameters_values=dict(
            scaler__with_mean=[True, False],
            scaler__with_std=[True, False],
            lin_reg__fit_intercept=[True, False]
        )
    )
]

# Configuring the Engine
For this demonstration, let's use ``r2_score`` to evaluate our modeling.

In [6]:
from sklearn.metrics import r2_score

from miraiml import Config

config = Config(
    local_dir='miraiml_local',
    problem_type='regression',
    score_function=r2_score,
    search_spaces=search_spaces,
    ensemble_id='Ensemble'
)

# Triggering the Engine
Let's also print the scores everytime the Engine finds a better solution.

In [7]:
from miraiml import Engine

def on_improvement(status):
    scores = status.scores
    for key in sorted(scores.keys()):
        print('{}: {}'.format(key, round(scores[key], 3)), end='; ')
    print()

engine = Engine(config=config, on_improvement=on_improvement)

Now we're ready to load the data

In [8]:
engine.load_train_data(train_data, 'SalePrice')
engine.load_test_data(test_data)

Let's see the status report.

In [9]:
from time import sleep

engine.restart()

sleep(1)

print('\nShuffling train data')
engine.shuffle_train_data(restart=True)

sleep(1)

engine.interrupt()

Exception in thread Thread-4:
Traceback (most recent call last):
  File "/opt/conda/lib/python3.6/threading.py", line 916, in _bootstrap_inner
    self.run()
  File "/opt/conda/lib/python3.6/threading.py", line 864, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/conda/lib/python3.6/site-packages/miraiml/main.py", line 555, in starter
    self.__main_loop__()
  File "/opt/conda/lib/python3.6/site-packages/miraiml/main.py", line 645, in __main_loop__
    self.train_data, self.train_target, self.test_data, self.config)
  File "/opt/conda/lib/python3.6/site-packages/miraiml/core.py", line 88, in predict
    model = self.model_class(**self.parameters)
  File "/opt/conda/lib/python3.6/site-packages/miraiml/core.py", line 583, in __init__
    self.set_params(**params)
  File "/opt/conda/lib/python3.6/site-packages/miraiml/core.py", line 614, in set_params
    'parameters are:\n' + ', '.join(allowed_params)
ValueError: Parameter lin_reg__fit_intercept is incompatible. The al


Shuffling train data


Exception in thread Thread-9:
Traceback (most recent call last):
  File "/opt/conda/lib/python3.6/threading.py", line 916, in _bootstrap_inner
    self.run()
  File "/opt/conda/lib/python3.6/threading.py", line 864, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/conda/lib/python3.6/site-packages/miraiml/main.py", line 555, in starter
    self.__main_loop__()
  File "/opt/conda/lib/python3.6/site-packages/miraiml/main.py", line 645, in __main_loop__
    self.train_data, self.train_target, self.test_data, self.config)
  File "/opt/conda/lib/python3.6/site-packages/miraiml/core.py", line 88, in predict
    model = self.model_class(**self.parameters)
  File "/opt/conda/lib/python3.6/site-packages/miraiml/core.py", line 583, in __init__
    self.set_params(**params)
  File "/opt/conda/lib/python3.6/site-packages/miraiml/core.py", line 614, in set_params
    'parameters are:\n' + ', '.join(allowed_params)
ValueError: Parameter lin_reg__fit_intercept is incompatible. The al

# Engine’s status analysis

In [10]:
status = engine.request_status()

In [11]:
print(status.build_report(include_features=True))

########################
best id: k-NeighborsRegressor
best score: 0.4338936274257381
########################
all scores:
    k-NeighborsRegressor: 0.4338936274257381
########################
id: Pipeline
model class: MiraiPipeline
n features: 2
parameters:
    lin_reg__fit_intercept: False
    scaler__with_mean: True
    scaler__with_std: False
features: TotRmsAbvGrd, YearBuilt
########################
id: k-NeighborsRegressor
model class: KNeighborsRegressor
n features: 2
parameters:
    n_neighbors: 5
    p: 3
    weights: distance
features: TotRmsAbvGrd, YearBuilt



# Final